In [1]:
import os
import sys
import pickle
import re
from pathlib import Path
import numpy as np
import pandas as pd

ADS_DEADLINE = "2022-03-25T11:00:00Z"
MAX_PROPOSALS = 25
MAX_REVIEWERS = 100
MAX_PAPERS = 25
LOOKBACK_YRS = 5
FIRST_AUTHOR_ONLY = False

LDA_NUM_TOPICS = 50
LDA_TOPIC_COUNTS = (15, 25, 50, 75)
EMBED_TOP_K = 5

EMBEDDING_MODELS = (
    "KaLM-Embedding/KaLM-embedding-multilingual-mini-instruct-v2.5",
    "IEITYuan/Yuan-embedding-2.0-en",
)

REPO_ROOT = next(
    path for path in (Path.cwd(), Path.cwd().parent)
    if (path / "scripts" / "score_matrices.py").exists()
)
sys.path.insert(0, str(REPO_ROOT))
from scripts.fetch_ads_papers import build_ads_synthetic_dataset, fetch_papers_ads
from scripts import score_matrices as cs

np.random.seed(11)
matrices = {}
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

/home/vicenteamado/.conda/envs/expertise/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-10-20 20:45:56.997172: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-10-20 20:45:56.997260: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-10-20 20:45:56.998678: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-10-20 20:45:57.007180: I tensorflow/core/platform/cpu_feature_guard.cc:182] This Tens

'false'

In [2]:
os.environ.setdefault("ADS_DEV_KEY", "t0WA0Y7ki1XR3wz9VxjmkAehyD6JsTAyDm7szZek") 

't0WA0Y7ki1XR3wz9VxjmkAehyD6JsTAyDm7szZek'

In [3]:
if "ADS_DEV_KEY" not in os.environ:
    raise EnvironmentError("Set ADS_DEV_KEY in the environment before running ADS-backed cells.")

## 1. Load data

In [4]:
def prepare_hst_jwst_data():
    dataset = build_ads_synthetic_dataset(
        deadline=ADS_DEADLINE,
        max_proposals=MAX_PROPOSALS,
        max_reviewers=MAX_REVIEWERS,
        papers_per_reviewer=MAX_PAPERS,
    )

    proposal_ids = [proposal["proposal_id"] for proposal in dataset["proposals"]]
    proposal_abstracts = [proposal["abstract"] or "" for proposal in dataset["proposals"]]
    proposal_texts = [
        f"TITLE: {proposal['title']}\nABSTRACT: {proposal['abstract'] or ''}"
        for proposal in dataset["proposals"]
    ]

    reviewer_ids = dataset["reviewers"]
    ground_truth = {
        proposal["proposal_id"]: proposal["first_author"]
        for proposal in dataset["proposals"]
        if proposal["first_author"] in reviewer_ids
    }
    reviewer_papers = []
    reviewer_paper_abstracts = []
    for rid in reviewer_ids:
        papers = dataset["reviewer_papers"].get(rid, [])
        # Preserve the reviewer axis when ADS returns no papers.
        # The empty document is a neutral profile for downstream models.
        if not papers:
            papers = [{"title": "", "abstract": ""}]
        reviewer_papers.append([
            (paper.get('title') or '') + (paper.get('abstract') or '')
            for paper in papers
        ])
        reviewer_paper_abstracts.append([
            paper.get("abstract") or "" for paper in papers
        ])

    return {
        "proposal_ids": proposal_ids,
        "proposal_abstracts": proposal_abstracts,
        "proposal_texts": proposal_texts,
        "reviewer_ids": reviewer_ids,
        "reviewer_papers": reviewer_papers,
        "reviewer_paper_abstracts": reviewer_paper_abstracts,
        "ground_truth": ground_truth,
    }

In [5]:
data = prepare_hst_jwst_data()

proposal_ids = data["proposal_ids"]
ground_truth = data["ground_truth"]
proposal_abstracts = data["proposal_abstracts"]
proposal_texts = data["proposal_texts"]
reviewer_ids = data["reviewer_ids"]
reviewer_papers = data["reviewer_papers"]
reviewer_paper_abstracts = data["reviewer_paper_abstracts"]

print(f"{len(proposal_ids)} proposals, {len(reviewer_ids)} reviewers")
print(f"reviewer paper histories: {sum(map(len, reviewer_papers))} papers")

25 proposals, 16 reviewers
reviewer paper histories: 248 papers


## 2. Compute baseline score matrices

These are the same baseline methods used by analyze_eso.ipynb. All implemented scoring methods and ablations run automatically on the HST/JWST synthetic dataset.

In [6]:
scores_tfidf = cs.tfidf(proposal_texts, reviewer_papers)
scores_tfidf_max_sim = cs.tfidf(
    proposal_texts, reviewer_papers, aggregation="max_sim")
matrices["TF-IDF"] = scores_tfidf
matrices["TF-IDF (max-sim)"] = scores_tfidf_max_sim

scores_lda = cs.lda(
    proposal_texts, reviewer_papers, num_topics=LDA_NUM_TOPICS)
matrices[f"LDA (K={LDA_NUM_TOPICS})"] = scores_lda

sbert_scores = {}
for aggregation in ("mean", "max", "max_sim", "topk_sim"):
    sbert_scores[aggregation] = cs.sentencetransformer(
        proposal_texts, reviewer_papers, aggregation=aggregation,
        top_k=EMBED_TOP_K)
    matrices[f"Sentence Transformer ({aggregation})"] = sbert_scores[aggregation]

specter_scores = {}
for aggregation in ("mean", "max", "max_sim", "topk_sim"):
    specter_scores[aggregation] = cs.specter(
        proposal_abstracts, reviewer_paper_abstracts,
        aggregation=aggregation, top_k=EMBED_TOP_K)
    matrices[f"SPECTER ({aggregation})"] = specter_scores[aggregation]

Fetching 4 files: 100%|██████████| 4/4 [00:00<00:00, 24105.20it/s]
There are adapters available but none are activated for the forward pass.
Fetching 4 files: 100%|██████████| 4/4 [00:00<00:00, 24818.37it/s]
There are adapters available but none are activated for the forward pass.
Fetching 4 files: 100%|██████████| 4/4 [00:00<00:00, 25266.89it/s]
There are adapters available but none are activated for the forward pass.
Fetching 4 files: 100%|██████████| 4/4 [00:00<00:00, 25771.45it/s]
There are adapters available but none are activated for the forward pass.


## 3. Publication-profile ablations

This is the same profile ablation as the original analysis: vary the maximum number of papers, lookback years, and first-author-only filtering while keeping the proposal set and reviewer IDs fixed.

The proposal records and anonymous reviewer roster are fixed from the initial ADS query; only reviewer publication histories are refetched.

In [7]:
PROFILE_CONFIGS = {
    "12 papers, 2 years": dict(max_papers=12, lookback_years=2, first_author_only=False),
    "50 papers, 10 years": dict(max_papers=50, lookback_years=10, first_author_only=False),
    "10 first-author papers, 5 years": dict(max_papers=10, lookback_years=5, first_author_only=True),
}

profile_reviewer_papers = {}
reviewer_lookup = {name: name for name in reviewer_ids}
for label, config in PROFILE_CONFIGS.items():
    fetched = fetch_papers_ads(
        reviewer_lookup,
        deadline=ADS_DEADLINE,
        **config,
    )
    papers = []
    for rid in reviewer_ids:
        reviewer_history = fetched.get(rid, [])
        if not reviewer_history:
            reviewer_history = [{"title": "", "abstract": ""}]
        papers.append([
            f"TITLE: {paper.get('title') or ''}\nABSTRACT: {paper.get('abstract') or ''}"
            for paper in reviewer_history
        ])
    profile_reviewer_papers[label] = papers
    matrices[f"TF-IDF ({label})"] = cs.tfidf(proposal_texts, papers)
    matrices[f"TF-IDF (max-sim; {label})"] = cs.tfidf(
        proposal_texts, papers, aggregation="max_sim")

    matrices[f"LDA ({label})"] = cs.lda(
        proposal_texts, papers, num_topics=LDA_NUM_TOPICS)

    matrices[f"Sentence Transformer (max-sim; {label})"] = cs.sentencetransformer(
        proposal_texts, papers, aggregation="max_sim", top_k=EMBED_TOP_K)


## 4. LDA topic-count sensitivity

This is the same LDA sensitivity sweep as the original analysis, run on the HST/JWST synthetic dataset.

In [8]:
lda_topic_ablation_scores = {}
for k in LDA_TOPIC_COUNTS:
    lda_topic_ablation_scores[k] = cs.lda(
        proposal_texts, reviewer_papers, num_topics=k)
    matrices[f"LDA (K={k})"] = lda_topic_ablation_scores[k]

## 5. Embedding-checkpoint sensitivity

This preserves the original checkpoint-sweep variables and calculation on the HST/JWST synthetic dataset.

In [9]:
embedding_checkpoint_scores = {}
for model_name in EMBEDDING_MODELS:
    embedding_checkpoint_scores[model_name] = cs.sentencetransformer(
        proposal_texts, reviewer_papers, model_name=model_name,
        aggregation="max_sim", top_k=EMBED_TOP_K)
    matrices[f"{model_name} (max-sim)"] = embedding_checkpoint_scores[model_name]

## 6. Lexical Analysts: TFIDF and SetenceTransformer

In [10]:
# This mirrors analyze_eso.ipynb exactly: fit TF-IDF on proposal texts only,
# remove the top 1% reviewer-profile features, and rescore masked proposals
# against the unchanged reviewer pool.
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize

TOP_FRACTION = 0.01  # analyze_eso.ipynb setting
vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2))
vectorizer.fit(proposal_texts)
terms = vectorizer.get_feature_names_out()
reviewer_documents = [" ".join(papers) for papers in reviewer_papers]
profiles = vectorizer.transform(reviewer_documents)

top_terms = {}
for rid, row in zip(reviewer_ids, profiles):
    n = max(1, int(np.ceil(row.nnz * TOP_FRACTION)))
    top_idx = row.indices[np.argsort(row.data)[-n:]]
    top_terms[rid] = terms[top_idx] if row.nnz else []

def remove_terms(text, words):
    for word in sorted(words, key=len, reverse=True):
        text = re.sub(rf"(?<!\w){re.escape(word)}(?!\w)", "", text, flags=re.I)
    return " ".join(text.split())

masked_texts = [
    remove_terms(text, top_terms.get(ground_truth[pid], []))
    for pid, text in zip(proposal_ids, proposal_texts)
]

# Recompute TF-IDF using the same fitted proposal-only vectorizer.
proposal_queries = normalize(vectorizer.transform(proposal_texts))
masked_queries = normalize(vectorizer.transform(masked_texts))
tfidf_original = (normalize(profiles) @ proposal_queries.T).toarray()
tfidf_masked = (normalize(profiles) @ masked_queries.T).toarray()
matrices["TF-IDF original (lexical ablation)"] = tfidf_original
matrices["TF-IDF designated masked"] = tfidf_masked

# Recompute SentenceTransformer scores against the unchanged paper pool.
sbert_masked = cs.sentencetransformer(
    masked_texts, reviewer_papers, aggregation="max_sim",
    top_k=EMBED_TOP_K)
matrices["Sentence Transformer designated masked"] = sbert_masked

def lexical_proposal_metrics(scores):
    scores_df = pd.DataFrame(scores, index=reviewer_ids, columns=proposal_ids)
    rows = []
    for proposal_id, reviewer_id in ground_truth.items():
        if proposal_id not in scores_df.columns or reviewer_id not in scores_df.index:
            continue
        col = scores_df[proposal_id].astype(float)
        rank = col.rank(ascending=False, method="average").loc[reviewer_id]
        others = col.drop(index=reviewer_id).to_numpy()
        std = others.std(ddof=1) if len(others) > 1 else np.nan
        z = ((col.loc[reviewer_id] - others.mean()) / std
             if len(others) and std > 0 else np.nan)
        rows.append({"rank": rank, "MRR": 1 / rank,
                     "Hit@5": int(rank <= 5), "z": z})
    return pd.DataFrame(rows)

lexical_scores = {
    "TF-IDF original": tfidf_original,
    "TF-IDF designated masked": tfidf_masked,
    "Sentence Transformer original": sbert_scores["max_sim"],
    "Sentence Transformer designated masked": sbert_masked,
}
lexical_ablation = pd.DataFrame({
    name: {
        "MRR": metrics["MRR"].mean(),
        "Hit@5": metrics["Hit@5"].mean(),
        "Median rank": metrics["rank"].median(),
        "Mean rank": metrics["rank"].mean(),
        "Mean z": metrics["z"].mean(),
        "N": len(metrics),
    }
    for name, matrix in lexical_scores.items()
    for metrics in [lexical_proposal_metrics(matrix)]
}).T
display(lexical_ablation)

,MRR,Hit@5,Median rank,Mean rank,Mean z,N
TF-IDF original,0.435568,0.64,5.0,6.26,1.641153,25.0
TF-IDF designated masked,0.270490,0.56,5.0,7.34,0.324306,25.0
Sentence Transformer original,0.298438,0.44,8.0,7.92,-0.150404,25.0
Sentence Transformer designated masked,0.289892,0.36,7.0,8.00,-0.167911,25.0


In [11]:
def proposal_metrics(scores, truth, reviewer_ids, proposal_ids, k_values=(5,)):
    scores_df = pd.DataFrame(scores, index=reviewer_ids, columns=proposal_ids)
    rows = []
    for proposal_id, reviewer_id in truth.items():
        
        col = scores_df[proposal_id].astype(float)
        rank = col.rank(ascending=False, method="average").loc[reviewer_id]
        others = col.drop(index=reviewer_id).to_numpy()
        std = others.std(ddof=1) if len(others) > 1 else np.nan
        z = ((col.loc[reviewer_id] - others.mean()) / std
             if len(others) and std > 0 else np.nan)
        row = {"proposal_id": proposal_id, "rank": rank,
               "MRR": 1 / rank, "z": z}
        for k in k_values:
            row[f"Hit@{k}"] = int(rank <= k)
        rows.append(row)
    return pd.DataFrame(rows).set_index("proposal_id")

all_metrics = {
    name: proposal_metrics(matrix, ground_truth, reviewer_ids, proposal_ids)
    for name, matrix in matrices.items()
}

evaluation_summary = pd.DataFrame({
    name: {
        "MRR": metrics["MRR"].mean(),
        "Hit@5": metrics["Hit@5"].mean(),
        "Median rank": metrics["rank"].median(),
        "Mean z": metrics["z"].mean(),
        "N": len(metrics),
    }
    for name, metrics in all_metrics.items()
}).T.sort_values("Hit@5", ascending=False)

display(evaluation_summary)

,MRR,Hit@5,Median rank,Mean z,N
"TF-IDF (12 papers, 2 years)",0.459000,0.84,4.0,1.410051,25.0
"TF-IDF (max-sim; 50 papers, 10 years)",0.438879,0.68,3.0,1.301748,25.0
"TF-IDF (max-sim; 12 papers, 2 years)",0.459154,0.68,4.0,1.536102,25.0
TF-IDF original (lexical ablation),0.435568,0.64,5.0,1.641153,25.0
"TF-IDF (10 first-author papers, 5 years)",0.421802,0.64,3.5,1.461108,25.0
TF-IDF (max-sim),0.425455,0.64,2.0,1.320964,25.0
"TF-IDF (max-sim; 10 first-author papers, 5 years)",0.436987,0.60,3.5,1.605489,25.0
TF-IDF,0.524566,0.56,3.0,1.775475,25.0
TF-IDF designated masked,0.270490,0.56,5.0,0.324306,25.0
LDA (K=15),0.255688,0.56,4.0,0.469158,25.0
